# Collecting the EarthPortal data

This notebook downloads everything the audit needs and saves it, untouched, in `data/raw/<RUN_DATE>/`:

- **I.** the catalogue: every public entry of EarthPortal;
- **II.** the portal's own metadata schema: the fields EarthPortal stores for each ontology;
- **III.** the latest submission of every ontology, with all its metadata fields;
- **IV.** the O'FAIRe assessment of every ontology, question by question.

It needs the API key written in the `.env` file at the root of the repository. The other notebooks
only read the files saved here, so they can be run again later on exactly the same data.

In [1]:
# Imports
import json
import time

import pandas as pd
import requests

from config import *

In [2]:
# We check that the API key is there, then prepare a session that sends it in a header
assert APIKEY, "EARTHPORTAL_APIKEY is empty: write your key in the .env file at the root of the repository"

session = requests.Session()
session.headers.update({"Authorization": f"apikey token={APIKEY}", "Accept": "application/json"})
QUIET = "display_links=false&display_context=false"     # shorter answers: no links, no JSON-LD context


def get(path, tries=3):
    """We ask the API for one path and try again, up to three times, if the network fails."""
    for attempt in range(1, tries + 1):
        try:
            answer = session.get(API_URL + path, timeout=60)
            answer.raise_for_status()
            return answer.json()
        except requests.RequestException as error:
            if attempt == tries:
                raise
            print(f"Retrying {path} after: {error}")
            time.sleep(3)


def save(obj, name):
    with open(RAW / name, "w", encoding="utf-8") as file:
        json.dump(obj, file, ensure_ascii=False, indent=1)
    print(f"Saved {(RAW / name).relative_to(REPO)}")

# I. The catalogue

In [3]:
# We ask for all public entries, views included
catalogue = get(f"/ontologies?include=all&also_include_views=true&{QUIET}")
save(catalogue, "catalogue.json")
print(f"{len(catalogue)} entries in the catalogue")
pd.DataFrame(catalogue)[["acronym", "name", "viewingRestriction"]].head()

Saved earthportal_sa_audit/data/raw/2026-10-02/catalogue.json
92 entries in the catalogue


,acronym,name,viewingRestriction
0,CALATHE,Cadastre and Land Administration Thesaurus,public
1,ENVTHES,Environmental Thesaurus,public
2,N4E,NFDI4Earth Ontology,public
3,NVS_C17,ICES Platform Codes,public
4,NVS_C19,SeaVoX salt and fresh water body gazetteer,public


# II. The portal's own metadata schema

In [4]:
# We ask which metadata fields the portal stores: on each submission (a version of an ontology)
# and on the ontology record itself
schema = {"submission": get(f"/submission_metadata?{QUIET}"),
          "ontology": get(f"/ontology_metadata?{QUIET}")}
save(schema, "metadata_schema.json")
print(f"{len(schema['submission'])} submission fields and {len(schema['ontology'])} ontology fields")
pd.DataFrame(schema["submission"])[["attribute", "namespace", "category"]].head()

Saved earthportal_sa_audit/data/raw/2026-10-02/metadata_schema.json
110 submission fields and 20 ontology fields


,attribute,namespace,category
0,submissionId,None,no
1,prefLabelProperty,None,object description properties
2,definitionProperty,None,object description properties
3,synonymProperty,None,object description properties
4,authorProperty,None,object description properties


# III. The latest submission of each ontology

In [5]:
# We download the latest submission of each ontology with all its fields (display=all)
submissions, failed = {}, []
for onto in catalogue:
    acronym = onto["acronym"]
    try:
        submissions[acronym] = get(f"/ontologies/{acronym}/latest_submission?display=all&{QUIET}")
    except requests.RequestException:
        submissions[acronym] = {}
        failed.append(acronym)
save(submissions, "submissions.json")
print(f"{len(submissions) - len(failed)} submissions downloaded; failed: {failed or 'none'}")

Saved earthportal_sa_audit/data/raw/2026-10-02/submissions.json
92 submissions downloaded; failed: none


# IV. The O'FAIRe assessment

EarthPortal's O'FAIRe service scores each ontology on 61 questions. It needs no API key when it is
called with `portal=earthportal`. We send the acronyms ten at a time to keep each answer small.

In [6]:
acronyms = [onto["acronym"] for onto in catalogue]
ofaire = {}
for start in range(0, len(acronyms), 10):
    batch = acronyms[start:start + 10]
    answer = requests.get(OFAIRE_URL, params={"portal": "earthportal", "ontologies": ",".join(batch)}, timeout=300)
    answer.raise_for_status()
    ofaire.update(answer.json().get("ontologies", {}))
save(ofaire, "ofaire.json")
missing = [a for a in acronyms if a not in ofaire]
print(f"{len(ofaire)} O'FAIRe results; missing: {missing or 'none'}")

Saved earthportal_sa_audit/data/raw/2026-10-02/ofaire.json
92 O'FAIRe results; missing: none


**Conclusion**: the snapshot of the day is saved in `data/raw/<RUN_DATE>/`. The next notebooks read it
from there; to work again on an older snapshot, write its date as `RUN_DATE` in `.env`.